# Metric Glance Span-Detection Classifier

Pipeline: export from D1 -> explore labels -> prepare features -> train -> export to ONNX.

The trained model will slot into `proposeSpans()` in `extension/converter.js` behind the
`useEncoder` flag. The arithmetic (conversion factors) is never touched by the model.

## Setup

```bash
# from the train/ directory
uv sync
uv run jupyter notebook classifier.ipynb
```

In [ ]:
# Print the path to the Python interpreter
!where python

In [ ]:
# Print environment information

# Import the os module
import os
import sys
import warnings


# Function to print environment information
def print_directory_info(target: str | None = None) -> None:
    """
    Prints the current working directory, active virtual environment, and Python version.
    If target is provided, navigates to that directory first.
    """
    # This is done in a function mainly to avoid cluttering the global namespace


    # Helper to make sure the working directory ends in a specific folder
    def _ensure_dir(target_path: str, current_path: str = os.getcwd()) -> str:
        """
        Recursively walks up the directory tree to find a folder matching target_path.
        Returns the full absolute path of the matched folder, or raises FileNotFoundError.
        """

        # If the current path is the target path, return the current path
        if os.path.basename(current_path) == target_path:
            return current_path

        # else, if the current path is the top level directory, return an error
        elif os.path.dirname(current_path) == current_path:
            raise FileNotFoundError(f"Could not find directory {target_path}")

        # In all other cases, call the function again with the parent directory
        else:
            return _ensure_dir(target_path, os.path.dirname(current_path))


    # Add warning if if target has spaces
    if target and " " in target:
        warnings.warn(
        "The target directory name contains spaces. GitHub repository names cannot include spaces. Spaces are typically replaced with hyphens (-) in the URL. This may cause confusion when pushing, pulling, or cloning. The current working directory is: \n" + os.getcwd()
        )

    # if a target is specified, change the current working directory to that folder
    if target:
        os.chdir(_ensure_dir(target))

    # Print a separator line
    print("-" * 100)

    # Print the current working directory
    current_directory: str = os.getcwd()
    parent_directory: str = os.path.dirname(current_directory)
    grandparent_directory: str = os.path.dirname(parent_directory)

    print(
        "Current working directory: "
        + f"{os.path.basename(grandparent_directory)}\\{os.path.basename(parent_directory)}\\{os.path.basename(current_directory)}"
    )

    # Check if a virtual environment is active
    venv_path: str | None = os.getenv("VIRTUAL_ENV")
    if venv_path:
        venv_name: str = os.path.basename(venv_path)
        _, dir_name = os.path.split(os.path.split(venv_path)[0])
        print(f"Environment: {dir_name}/\033[1m{venv_name}\033[0m")  # bold
    else:
        print("\033[1m\033[3m" + "No virtual environment" + "\033[0m")  # bold + italic

    # Print the Python version
    print("Python version: " + sys.version)

    # Print another separator line
    print("-" * 100)



# Call the function
print_directory_info("metric-glance")
# Replace "expected current directory name" with the actual name of the directory you expect to be in.

In [ ]:
import subprocess, json, pathlib
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

import onnx
import onnxruntime as rt
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType

# Resolve paths regardless of whether CWD is train/ or the repo root
# (an os.chdir to the repo root in another cell would break relative paths)
_cwd: pathlib.Path = pathlib.Path.cwd()
COLLECT_DIR: pathlib.Path
DATA_DIR: pathlib.Path
if (_cwd / "collect").exists():      # CWD is already the repo root
    COLLECT_DIR = _cwd / "collect"
    DATA_DIR    = _cwd / "train" / "data"
else:                                 # CWD is train/
    COLLECT_DIR = _cwd / ".." / "collect"
    DATA_DIR    = _cwd / "data"

COLLECT_DIR = COLLECT_DIR.resolve()
DATA_DIR    = DATA_DIR.resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"COLLECT_DIR: {COLLECT_DIR}")
print(f"DATA_DIR:    {DATA_DIR}")

---
## 1. Authenticate with Cloudflare and export from D1

`wrangler` needs an active Cloudflare session to hit the remote D1 database.
Run the cell below first. If it prints your email, you are ready. If it errors,
open a terminal in `../collect/` and run:

```bash
npx wrangler login
```

That opens a browser OAuth flow. Come back and re-run the cell once it finishes.

Alternatively, set the `CLOUDFLARE_API_TOKEN` environment variable to a token
that has D1 read permissions, and wrangler will use that instead of the browser flow.

In [ ]:
import sys as _sys

# Check wrangler authentication before doing anything else
collect_cwd: pathlib.Path = pathlib.Path(COLLECT_DIR)

if not collect_cwd.is_dir():
    found: pathlib.Path | None = None
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        candidate: pathlib.Path = base / "collect"
        if candidate.is_dir():
            found = candidate.resolve()
            break
    if found is None:
        raise FileNotFoundError(
            f"Could not find a valid collect/ directory. Current COLLECT_DIR: {COLLECT_DIR}"
        )
    COLLECT_DIR = found
    collect_cwd = found
else:
    collect_cwd = collect_cwd.resolve()

_flags: int = subprocess.CREATE_NO_WINDOW if _sys.platform == "win32" else 0

result: subprocess.CompletedProcess[str] = subprocess.run(
    "npx wrangler whoami",
    capture_output=True,
    text=True,
    encoding="utf-8",
    stdin=subprocess.DEVNULL,
    cwd=str(collect_cwd),
    shell=True,
    creationflags=_flags,
)
stdout: str = result.stdout or ""
stderr: str = result.stderr or ""
combined: str = f"{stdout}\n{stderr}".lower()

if result.returncode != 0 or "not authenticated" in combined:
    raise RuntimeError(
        "wrangler is not authenticated.\n"
        "Run this in a terminal from the collect/ directory:\n"
        "    npx wrangler login\n"
        "Then re-run this cell.\n\n"
        f"stdout:\n{stdout}\n\nstderr:\n{stderr}"
    )
if stdout.strip():
    print(stdout.strip())
elif stderr.strip():
    print(stderr.strip())
else:
    print("wrangler is authenticated.")

In [ ]:
import sys

# Pull fresh data from D1 on every run, then load it.
#
# The actual wrangler/Node call lives in train/refresh_data.py and runs as a
# SEPARATE process. The kernel only ever spawns python.exe here, never Node,
# so the Jupyter-on-Windows Node crash (0xC0000409) can at worst make this
# subprocess fail; it can never take down the kernel. On any failure we fall
# back to the last cached submissions.json.
#
# Set REFRESH_ON_RUN = False to skip the D1 pull and just use the cached file
# (offline, or to avoid the ~few-second wrangler round-trip).
REFRESH_ON_RUN: bool = True

EXPORT_FILE: pathlib.Path = DATA_DIR / "submissions.json"
REFRESH_SCRIPT: pathlib.Path = DATA_DIR.parent / "refresh_data.py"   # train/refresh_data.py


def _refresh_from_d1() -> tuple[bool, str]:
    """Run refresh_data.py in a separate process. Returns (ok, message)."""
    flags: int = subprocess.CREATE_NO_WINDOW if sys.platform == "win32" else 0
    try:
        p: subprocess.CompletedProcess[str] = subprocess.run(
            [sys.executable, str(REFRESH_SCRIPT)],
            capture_output=True, text=True, encoding="utf-8",
            stdin=subprocess.DEVNULL, creationflags=flags, timeout=180,
        )
    except Exception as e:  # never let a refresh problem abort the notebook
        return False, f"could not start refresh_data.py: {e}"
    if p.returncode != 0:
        return False, (p.stderr or p.stdout or f"exit {p.returncode}").strip()
    return True, (p.stdout or "").strip()


if REFRESH_ON_RUN:
    ok, msg = _refresh_from_d1()
    print(f"D1 refresh: {msg}" if ok else f"WARNING: refresh failed, using cached file.\n{msg}")

if not EXPORT_FILE.exists():
    raise FileNotFoundError(
        f"No data at {EXPORT_FILE} and the D1 refresh did not produce one.\n"
        "Run this from a terminal in the train/ directory to diagnose:\n"
        "    python refresh_data.py"
    )

rows: list[dict] = json.loads(EXPORT_FILE.read_text(encoding="utf-8"))
df_raw: pd.DataFrame = pd.DataFrame(rows)
print(f"Loaded {len(df_raw):,} rows from {EXPORT_FILE.name}  (shape {df_raw.shape})")
df_raw.head()

---
## 2. Trusted installs (sample weighting)

`install_id` groups rows by who submitted them (a random per-install id, not PII).
Corrections from trusted installs (e.g. your own) should count for more during
training. The trusted id(s) are read from a local, gitignored `.env` file, never
hardcoded, so they stay out of this open-source repo. See `.env.example` for how
to find yours.

In [ ]:
from dotenv import load_dotenv

# Load train/.env (gitignored). The real id(s) live there, not in this notebook.
load_dotenv(DATA_DIR.parent / ".env")

TRUSTED_INSTALL_IDS: set[str] = {
    s.strip() for s in os.getenv("MG_TRUSTED_INSTALL_IDS", "").split(",") if s.strip()
}

# Per-row weight: trusted installs count most, then corrected, then seen/auto.
TIER_WEIGHT: dict[str, float] = {"corrected": 3.0, "seen": 1.0, "auto": 0.5}


def row_weight(r: pd.Series) -> float:
    if TRUSTED_INSTALL_IDS and r.get("install_id") in TRUSTED_INSTALL_IDS:
        return 5.0
    return TIER_WEIGHT.get(r.get("tier"), 1.0)


if "install_id" not in df_raw.columns:
    print("NOTE: no install_id column yet. Re-run the data-load cell to pull it from D1.")
else:
    df_raw["sample_weight"] = df_raw.apply(row_weight, axis=1)
    n_trusted: int = int(df_raw["install_id"].isin(TRUSTED_INSTALL_IDS).sum()) if TRUSTED_INSTALL_IDS else 0
    # Print only aggregates, never the ids themselves (this output may be committed).
    print(f"Distinct installs:      {df_raw['install_id'].nunique():,}")
    print(f"Trusted ids configured: {len(TRUSTED_INSTALL_IDS)}")
    print(f"Rows from trusted:      {n_trusted:,}")
    print("\nWeight distribution:")
    print(df_raw["sample_weight"].value_counts().sort_index())